<h2>Tokenizer<h2\>

In [78]:
import torch
import pandas as pd

class CharTokenizer:
    def __init__(self, text=None, pad_token='<PAD>', unk_token='<UNK>'):
        self.pad_token = pad_token
        self.unk_token = unk_token
        self.char2idx = {}
        self.idx2char = {}
        self.vocab_built = False

        if text:
            self.build_vocab(text)
        
        self.vocab_size = len(self.char2idx)

    def build_vocab(self, text):
        unique_chars = sorted(set(text))
        # Reserve indices for PAD and UNK
        self.char2idx = {self.pad_token: 0, self.unk_token: 1}
        for i, ch in enumerate(unique_chars, start=2):
            self.char2idx[ch] = i
        self.idx2char = {i: ch for ch, i in self.char2idx.items()}
        self.vocab_built = True

    def encode(self, text, max_length=None):
        if not self.vocab_built:
            raise ValueError("Vocabulary not built yet. Call build_vocab first.")

        encoded = [self.char2idx.get(ch, self.char2idx[self.unk_token]) for ch in text]

        # Truncate if needed
        if max_length is not None:
            encoded = encoded[:max_length]

        # Pad if needed
        if max_length is not None and len(encoded) < max_length:
            pad_length = max_length - len(encoded)
            encoded += [self.char2idx[self.pad_token]] * pad_length

        return torch.tensor(encoded, dtype=torch.long)

    def decode(self, indices):
        # Accept either tensor or list
        if isinstance(indices, torch.Tensor):
            indices = indices.tolist()
        chars = [self.idx2char.get(i, self.unk_token) for i in indices]
        # Strip padding tokens from end
        while chars and chars[-1] == self.pad_token:
            chars.pop()
        return ''.join(chars)
    
    def create_mask(self, encoded_tensor):
        pad_token_idx = self.char2idx[self.pad_token]
        return (encoded_tensor != pad_token_idx).long()


<h2> Dataset <h2\>

In [79]:
from torch.utils.data import Dataset

class CISC2010DataSet(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=1024):
        self.df = dataframe.drop_duplicates(subset='content', keep='first')
        self.df = self.df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]
        text = row["content"]
        label = int(row["classification"])

        encoded = self.tokenizer.encode(text, max_length=self.max_len)
        mask = self.tokenizer.create_mask(encoded)

        return encoded, mask, torch.tensor(label, dtype=torch.long)

<h2> Postional encoding <h2\>

In [80]:
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=1024):
        super().__init__()
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim)
        )
        pe[:, 0::2] = torch.sin(position * div_term)  # even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # odd indices
        pe = pe.unsqueeze(0)  # shape: (1, max_len, embed_dim)
        self.register_buffer("pe", pe)

    def forward(self, x):
        """
        Args:
            x: Tensor, shape [batch_size, seq_len, embed_dim]
        """
        x = x + self.pe[:, :x.size(1)]
        return x


<h2> The Model <h2\>

In [81]:
class TransformerBased(nn.Module):
    def __init__(self, input_dim, embed_dim, num_heads, output_dim, device, num_layers : int = 1, dropout=0.1, max_len=1024):
        super().__init__()

        assert embed_dim % num_heads == 0 # each head should take the same number of features therfore the num_heads should subtract the embed_dim

        self.embedding = nn.Embedding(input_dim, embed_dim)
        self.pos_encoding = PositionalEncoding(embed_dim, max_len=max_len)

        self.layers = nn.ModuleList([
            nn.MultiheadAttention(
                embed_dim=embed_dim,
                num_heads=num_heads,
                dropout=dropout,
                batch_first=True,
                device = device
            ) for _ in range(num_layers)
        ])

        self.attn_norm = nn.ModuleList([nn.LayerNorm(embed_dim) for _ in range(num_layers)])

        self.ffnn = nn.ModuleList(
            [nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim)
            )for i in range(num_layers)]
        )

        self.ffn_norm = nn.ModuleList([nn.LayerNorm(embed_dim) for _ in range(num_layers)])

        self.classifier = nn.Linear(embed_dim, output_dim, bias=True)

    def forward(self, x, attn_mask=None):

        # Embedding + Positional Encoding
        x = self.embedding(x)
        x = self.pos_encoding(x)

        # Convert mask for MultiheadAttention (needs True for ignore positions)
        if attn_mask is not None:
            # attn_mask: (batch, seq_len) → key_padding_mask: (batch, seq_len)
            key_padding_mask = attn_mask == 0
        else:
            key_padding_mask = None

        # Self-attention and the normalization + ffnn
        for idx, attention in enumerate(self.layers):
            #attn
            attn_out, _ = attention(
            x, x, x, key_padding_mask=key_padding_mask
            )

            #ffnn
            x = self.attn_norm[idx](x + attn_out)
            ffn_out = self.ffnn[idx](x)
            x = self.ffn_norm[idx](x + ffn_out)

        # Pooling (mean pooling over valid tokens if mask provided)
        if attn_mask is not None:
            lengths = attn_mask.sum(dim=1, keepdim=True)
            pooled = (x * attn_mask.unsqueeze(-1)).sum(dim=1) / lengths
        else:
            pooled = x.mean(dim=1)

        logits = self.classifier(pooled)
        return logits

<h2> train function <h2\>

In [82]:
import os
from tqdm.notebook import tqdm

def train(model,
              dataloader,
              optimizer,
              device,
              loss_function,
              epochs: int = 1000,
              save_every: int = 100,
              save_dir: str = "checkpoints"):
    os.makedirs(save_dir, exist_ok=True)
    model.to(device)

    start_epoch = 1
    ckpts = [f for f in os.listdir(save_dir) if f.endswith(".pt")]
    if ckpts:
        latest_ckpt = max(ckpts, key=lambda x: int(x.split("_")[-1].split(".")[0]))
        ckpt_path = os.path.join(save_dir, latest_ckpt)
        checkpoint = torch.load(ckpt_path, map_location=device)
        
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        start_epoch = checkpoint["epoch"] + 1
        
        tqdm.write(f"✅ Loaded checkpoint '{ckpt_path}' (epoch {checkpoint['epoch']})")
    

    epoch_bar = tqdm(range(start_epoch, epochs + 1), desc="Epochs", unit="epoch", leave=False)

    for epoch in epoch_bar:
        model.train()
        running_loss = 0.0

        for input, masks, labels in dataloader:

            input = input.to(device)
            labels = labels.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()
            logits = model(input, masks)
            loss = loss_function(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_loss = running_loss / len(dataloader)
        epoch_bar.set_postfix(avg_loss=f"{avg_loss:.4f}")

        # ── checkpoint every `save_every` epochs ───────────────────────────
        if epoch % save_every == 0:
            ckpt_path = os.path.join(save_dir, f"pln_epoch_{epoch}.pt")
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                },
                ckpt_path,
            )
            tqdm.write(f"✓ Saved checkpoint → {ckpt_path}")
        


<h2> eval function <h2\>

In [83]:
from sklearn.metrics import precision_score, recall_score, f1_score
from tqdm import tqdm
import torch.nn.functional as F

def evaluate(model, dataloader, device, tokenizer, max_samples):
    model.eval()
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    correct_samples = []
    incorrect_samples = []

    batch_bar = tqdm(enumerate(dataloader), total=len(dataloader), unit="batch", leave=False)
    with torch.no_grad():
        for batch_idx, (inputs, masks, labels) in batch_bar:
            inputs = inputs.to(device)
            labels = labels.to(device)
            masks = masks.to(device)
        
            batch_bar.set_description(f"Batch {batch_idx}")

            logits = model(inputs, masks)
            probs = F.softmax(logits, dim=-1)
            preds = probs.argmax(dim=-1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

            correct_mask = (preds == labels)
            correct += correct_mask.sum().item()
            total += labels.size(0)

            for i in range(len(labels)):
                if len(correct_samples) >= max_samples and len(incorrect_samples) >= max_samples:
                    break
                decoded_input = tokenizer.decode(inputs[i].cpu())
                true_label = labels[i].item()
                pred_label = preds[i].item()

                sample = {
                    'input': decoded_input,
                    'true_label': true_label,
                    'pred_label': pred_label,
                }

                if pred_label == true_label and len(correct_samples) < max_samples:
                    correct_samples.append(sample)
                elif pred_label != true_label and len(incorrect_samples) < max_samples:
                    incorrect_samples.append(sample)

            if len(correct_samples) >= max_samples and len(incorrect_samples) >= max_samples:
                break

    accuracy = correct / total if total > 0 else 0
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)


    return accuracy, precision, recall, f1, correct_samples, incorrect_samples

<h1> The grand fianle <h1\>

In [ ]:
from torch.utils.data import DataLoader
import torch.optim as optim

def main():

    max_len = 200

    # Load dataset
    train_csv_path = "CISC2010_cleaned_train.csv"
    train_df = pd.read_csv(train_csv_path)

    test_csv_path = "CISC2010_cleaned_train.csv"
    test_df = pd.read_csv(test_csv_path)

    # Build tokenizer vocab from all content
    tokenizer = CharTokenizer("".join(train_df["content"].tolist()))
    #print("cheching the tokanization of 1 of the failures: ", tokenizer.encode("http://localhost:8080/tienda1/publico/registro.jsp?modo=registro&login=sarge&password=apad-r8nad5ra&"))

    # Dataset + DataLoader
    train_dataset = CISC2010DataSet(train_df, tokenizer, max_len=max_len)
    train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    test_dataset = CISC2010DataSet(test_df, tokenizer, max_len=max_len)
    test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"running on {device}")

    # Init model
    vocab_size = tokenizer.vocab_size
    embedding_dim = 64
    num_heads = 8
    model = TransformerBased(input_dim=vocab_size, embed_dim=embedding_dim, num_heads=num_heads, max_len=max_len, dropout = 0.1, device=device, output_dim=2, num_layers=4)

    # Optimizer
    print(f"{sum(p.numel() for p in model.parameters() if p.requires_grad)} parameters to optimaize")
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    # Train
    train(model=model,
              dataloader=train_dataloader,
              optimizer=optimizer,
              device=device,
              loss_function=F.cross_entropy,
              epochs = 400,
              save_every = 50,
              save_dir = "checkpointsDROP1LAYER4")
    
    accuracy, precision, recall, f1, correct_samples, incorrect_samples = evaluate(
        model=model,
        dataloader=test_dataloader,
        device=device,
        tokenizer=tokenizer,
        max_samples=10
        )
    #encoded = tokenizer.encode("http://localhost:8080/tienda1/publico/registro.jsp?modo=registro&login=sarge&password=apad-r8nad5ra&", max_length=max_len)
    #mask = tokenizer.create_mask(encoded)
    #print("sample prediction: ", model(encoded.unsqueeze(0).to(device), mask.unsqueeze(0).to(device)).argmax(dim=-1))
    
    print("\n====== Final Evaluation Metrics ======")
    print(f"Accuracy   : {accuracy:.4f}")
    print(f"Precision  : {precision:.4f}")
    print(f"Recall     : {recall:.4f}")
    print(f"F1 Score   : {f1:.4f}")
    print("======================================\n")

    print("Sample Correct Predictions:")
    for sample in correct_samples:
        print(f"Input      : {sample['input']}")
        print(f"True Label : {sample['true_label']}, Predicted: {sample['pred_label']}")
        print("-----")

    print("\nSample Incorrect Predictions:")
    for sample in incorrect_samples:
        print(f"Input      : {sample['input']}")
        print(f"True Label : {sample['true_label']}, Predicted: {sample['pred_label']}")
        print("-----")

if __name__ == "__main__":
    main()

cheching the tokanization of 1 of the failures:  tensor([57, 69, 69, 65, 20,  9,  9, 61, 64, 52, 50, 61, 57, 64, 68, 69, 20, 18,
        10, 18, 10,  9, 69, 58, 54, 63, 53, 50, 11,  9, 65, 70, 51, 61, 58, 52,
        64,  9, 67, 54, 56, 58, 68, 69, 67, 64,  8, 59, 68, 65, 22, 62, 64, 53,
        64, 21, 67, 54, 56, 58, 68, 69, 67, 64,  4, 61, 64, 56, 58, 63, 21, 68,
        50, 67, 56, 54,  4, 65, 50, 68, 68, 72, 64, 67, 53, 21, 50, 65, 50, 53,
         7, 67, 18, 63, 50, 53, 15, 67, 50,  4])
running on cuda
204994 parameters to optimaize
✅ Loaded checkpoint 'checkpointsDROP1LAYER4/pln_epoch_400.pt' (epoch 400)


sample prediction:  tensor([1], device='cuda:0')

====== Final Evaluation Metrics ======
Accuracy   : 0.9993
Precision  : 0.9998
Recall     : 0.9991
F1 Score   : 0.9994

Sample Correct Predictions:
Input      : http://localhost:8080/tienda1/publico/entrar.jsp HTTP/1.1 errorMsg=Credenciales+incorrectas%27%3B+DROP+TABLE+usuarios%3B+SELECT+*+FROM+datos+WHERE+nombre+LIKE+%27%25
True Label : 1, Predicted: 1
-----
Input      : http://localhost:8080/tienda1/publico/productos.jsp HTTP/1.1 
True Label : 0, Predicted: 0
-----
Input      : http://localhost:8080/tienda1/miembros/editar.jsp HTTP/1.1 modo=registro&login=gantt&password=sae19r8&nombre=Libe&apellidos=Amusquivar&email=mccormack%40neotelecom.tn&dni=35453403F&direccion=Calle+Tam
True Label : 1, Predicted: 1
-----
Input      : http://localhost:8080/tienda1/global/titulo.jsp HTTP/1.1 
True Label : 0, Predicted: 0
-----
Input      : http://localhost:8080/tienda1/publico/pagar.jsp?modo=insertar&precio=5628&B1=Pasar+por+caja HTTP/1.1 
True Lab